# 02. 전처리 (Preprocessing)

---
### 0. 라이브러리 및 데이터 로드

In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 임포트 완료!")

라이브러리 임포트 완료!


In [30]:
# 데이터 로드
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

# ID와 타겟 분리
train_id = train['ID']
test_id = test['ID']
target = train['withdrawal']

# ID와 타겟 제거
train = train.drop(['ID', 'withdrawal'], axis=1)
test = test.drop(['ID'], axis=1)

print(f"피처 수: {train.shape[1]}")

Train: (1056, 46), Test: (788, 45)
피처 수: 44


---
### 1. 불필요한 컬럼 제거 및 변환

In [31]:
# drop할 컬럼 목록
drop_cols = [
    'generation', 'nationality', 'inflow_route',
    'desired_career_path', 'desired_job',
    'certificate_acquisition', 'desired_certificate', 'certificate_study_period',
    'desired_job_except_data',
    'incumbents_level', 'incumbents_lecture', 'incumbents_company_level',
    'incumbents_lecture_type', 'incumbents_lecture_scale', 'incumbents_lecture_scale_reason',
    'interested_company', 'expected_domain', 'onedayclass_topic',
]

print(f"제거할 컬럼 수: {len(drop_cols)}개")

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

print(f"제거 후 Train 크기: {train.shape}")
print(f"제거 후 Test 크기: {test.shape}")

제거할 컬럼 수: 18개
제거 후 Train 크기: (1056, 26)
제거 후 Test 크기: (788, 26)


In [32]:
# major_field: 쉼표 제거 후 첫 번째 값만 남기기
train['major_field'] = train['major_field'].str.replace(',', '').str.split().str[0]
test['major_field'] = test['major_field'].str.replace(',', '').str.split().str[0]

print(train['major_field'].value_counts())

major_field
IT       378
공학       224
자연과학     129
경영학      119
사회과학      76
경제통상학     72
인문학       27
의약학        9
교육학        7
예체능        5
법학         2
Name: count, dtype: int64


In [33]:
# 희소 값 통합: 빈도 30 미만인 값을 'Other'로 통합
rare_merge_cols = ['school1', 'major_field', 'major1_1', 'major1_2', 'what_to_gain']

for col in rare_merge_cols:
    counts = train[col].value_counts()
    rare_values = counts[counts < 30].index
    print(f"[{col}] 통합 대상 ({len(rare_values)}개)")
    train[col] = train[col].replace(rare_values, 'Other')
    test[col] = test[col].replace(rare_values, 'Other')
    print(f"  -> 통합 후 고유값: {train[col].nunique()}개\n")

[school1] 통합 대상 (62개)
  -> 통합 후 고유값: 16개

[major_field] 통합 대상 (5개)
  -> 통합 후 고유값: 7개

[major1_1] 통합 대상 (4개)
  -> 통합 후 고유값: 8개

[major1_2] 통합 대상 (7개)
  -> 통합 후 고유값: 5개

[what_to_gain] 통합 대상 (5개)
  -> 통합 후 고유값: 5개



In [34]:
# NA 여부를 boolean으로 변환 (값이 있으면 True, NA면 False)
bool_cols = ['contest_award', 'contest_participitation', 'idea_contest']

for col in bool_cols:
    train[col] = train[col].notna()
    test[col] = test[col].notna()

print(train[bool_cols].dtypes)

contest_award              bool
contest_participitation    bool
idea_contest               bool
dtype: object


In [35]:
# previous_class_* -> 개별 수업 binary 컬럼으로 변환
prev_cols = ['previous_class_3', 'previous_class_4', 'previous_class_5',
             'previous_class_6', 'previous_class_7']

# 모든 previous_class 컬럼에서 개별 수업명 추출
all_classes = set()
for col in prev_cols:
    for val in train[col].dropna():
        for v in val.split(','):
            all_classes.add(v.strip())
class_names = sorted(all_classes)

# 각 수업별 수강 여부 binary 컬럼 생성
for cls in class_names:
    train[cls] = False
    test[cls] = False
    for col in prev_cols:
        train.loc[train[col].str.contains(cls, na=False), cls] = True
        test.loc[test[col].str.contains(cls, na=False), cls] = True

train = train.drop(columns=prev_cols)
test = test.drop(columns=prev_cols)

print(f"생성된 컬럼 ({len(class_names)}개): {class_names}")

생성된 컬럼 (7개): ['R기초', 'SQL', '데분기', '데분중', '파문기', '파문응', '해당없음']


In [36]:
# 결측치 100개 이상인 컬럼 제거
missing_counts = train.isnull().sum()
high_missing_cols = missing_counts[missing_counts >= 100].index.tolist()

print(f"제거할 컬럼 ({len(high_missing_cols)}개): {high_missing_cols}")

train = train.drop(columns=high_missing_cols)
test = test.drop(columns=high_missing_cols)

print(f"제거 후 Train 크기: {train.shape}")
print(f"제거 후 Test 크기: {test.shape}")

제거할 컬럼 (4개): ['major1_2', 'class2', 'class3', 'class4']
제거 후 Train 크기: (1056, 24)
제거 후 Test 크기: (788, 24)


---
### 2. 결측치 처리 및 인코딩

In [37]:
# 남은 결측치 확인
print("남은 결측치:")
missing = train.isnull().sum()
print(missing[missing > 0])

# 수치형: 중앙값, 범주형: 최빈값으로 채우기
numerical_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train.select_dtypes(include=['object']).columns.tolist()

for col in numerical_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

for col in categorical_cols:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

print(f"\n결측치 처리 후 남은 결측치: {train.isnull().sum().sum()}개")

남은 결측치:
major type             7
major1_1               5
what_to_gain          11
hope_for_group        11
major_field            8
completed_semester    84
project_type          11
dtype: int64

결측치 처리 후 남은 결측치: 0개


In [38]:
# Label Encoding
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)}개 카테고리")

print(f"\n인코딩 완료! 총 {len(label_encoders)}개 컬럼")
print(f"\n데이터 타입:\n{train.dtypes.value_counts()}")

  school1: 16개 카테고리
  major type: 2개 카테고리
  major1_1: 8개 카테고리
  job: 4개 카테고리
  re_registration: 2개 카테고리
  whyBDA: 7개 카테고리
  what_to_gain: 5개 카테고리
  hope_for_group: 3개 카테고리
  major_field: 7개 카테고리
  completed_semester: 2개 카테고리
  project_type: 2개 카테고리

인코딩 완료! 총 11개 컬럼

데이터 타입:
int64      12
bool       11
float64     1
Name: count, dtype: int64


---
### 3. Train/Test 분리 및 저장

In [39]:
# 피처와 타겟
X = train
y = target

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny 분포:\n{y.value_counts()}")

X shape: (1056, 24)
y shape: (1056,)

y 분포:
withdrawal
1    730
0    326
Name: count, dtype: int64


In [40]:
# 학습/검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

X_train: (844, 24), y_train: (844,)
X_val: (212, 24), y_val: (212,)


In [41]:
# 전처리된 데이터 저장
train.to_csv('./data/train_processed.csv', index=False)
test.to_csv('./data/test_processed.csv', index=False)

pd.DataFrame({'ID': train_id, 'withdrawal': target}).to_csv('./data/train_target.csv', index=False)
pd.DataFrame({'ID': test_id}).to_csv('./data/test_id.csv', index=False)

print("전처리된 데이터 저장 완료!")
print("  - train_processed.csv")
print("  - test_processed.csv")
print("  - train_target.csv")
print("  - test_id.csv")

전처리된 데이터 저장 완료!
  - train_processed.csv
  - test_processed.csv
  - train_target.csv
  - test_id.csv


---
## 요약

1. 불필요한 컬럼 제거 (18개)
2. 피처 변환 (major_field, 희소값 통합, boolean 변환, previous_class 분리)
3. 결측치 100개 이상 컬럼 제거
4. 남은 결측치 처리 (수치형: 중앙값, 범주형: 최빈값)
5. Label Encoding
6. 데이터 저장